# ALS parameter search


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
import time
from itertools import product
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

CWD = Path.cwd()
ROOT_DIR = CWD if (CWD / "src").exists() else CWD.parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit
from src.metrics.metrics import recall_at_k, ndcg_at_k

DATA_DIR = ROOT_DIR / "notebook" / "data" / "11092026"
if not DATA_DIR.exists():
    DATA_DIR = CWD / "data" / "11092026"


## Data


In [ ]:
df_interactions = pd.read_parquet(DATA_DIR / "interactions.parquet")
encoders = joblib.load(DATA_DIR / "encoders.pkl")
metadata = joblib.load(DATA_DIR / "metadata.pkl")

user_encoder = encoders["user_encoder"]
n_users = metadata["n_users"]

df_interactions = df_interactions.copy()
df_interactions["user_id"] = user_encoder.inverse_transform(df_interactions["user_idx"])
df_interactions["movie_id"] = encoders["movie_encoder"].inverse_transform(df_interactions["movie_idx"])


In [ ]:
splitter = TemporalSplit()
train, val, test = splitter.split_to_three(df_interactions)
rank_history, rank_target = splitter.split_to_two(train)

test_history = pd.concat([train, val], ignore_index=True)
test_target = test.copy()


## ALS


In [ ]:
def fit_als(
    history,
    n_users,
    factors=25,
    regularization=0.3,
    iterations=30
):
    movie_ids = history["movie_id"].unique()
    movie_to_als = {
        movie_id: idx
        for idx, movie_id in enumerate(movie_ids)
    }

    als_to_movie = {
        idx: movie_id
        for movie_id, idx in movie_to_als.items()
    }

    positive = history.loc[
        history["rating"] >= 4,
        ["user_idx", "movie_id", "rating"]
    ].copy()

    positive["als_movie_idx"] = positive["movie_id"].map(movie_to_als)
    values = (positive["rating"] - 3).values.astype(np.float32)

    user_item = csr_matrix(
        (
            values,
            (positive["user_idx"], positive["als_movie_idx"])
        ),
        shape=(n_users, len(movie_to_als)),
        dtype=np.float32
    )

    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        num_threads=6,
        random_state=42,
        alpha=1
    )

    model.fit(user_item, show_progress=True)

    return {
        "model": model,
        "user_item": user_item,
        "movie_to_als": movie_to_als,
        "als_to_movie": als_to_movie
    }


In [ ]:
def build_candidates(users, history, als_artifacts, k=500):
    seen = history.groupby("user_idx")["movie_id"].apply(set).to_dict()

    als_model = als_artifacts["model"]
    user_item = als_artifacts["user_item"]
    als_to_movie = als_artifacts["als_to_movie"]

    rows = []

    for user_idx in users:

        candidates = {}
        watched = seen.get(user_idx, set())

        request_k = min(k + len(watched), user_item.shape[1])
        ids, scores = als_model.recommend(
            userid=int(user_idx),
            user_items=user_item[int(user_idx)],
            N=request_k,
            filter_already_liked_items=True,
        )

        for movie_idx, score in zip(ids, scores):
        
            movie_id = als_to_movie[int(movie_idx)]

            if movie_id in watched:
                continue

            candidates[movie_id] = max(candidates.get(movie_id, score), score)

            if len(candidates) >= k:
                break

        for movie_id, score in candidates.items():
            rows.append(
                {
                    "user_idx": user_idx,
                    "movie_id": movie_id,
                    "score": score
                }
            )

    result = pd.DataFrame(rows)
    result["user_id"] = user_encoder.inverse_transform(result["user_idx"])

    return result


In [ ]:
def recall(candidates, target, min_rating=4):

    relevant = (
        target.loc[target["rating"] >= 4]
        .groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    predicted = candidates.groupby("user_id")["movie_id"].apply(set).to_dict()

    recalls = []

    for user_id, y_true in relevant.items():
        if not y_true:
            continue

        y_pred = predicted.get(user_id, set())
        recalls.append(len(y_true & y_pred) / len(y_true))

    return np.mean(recalls) if recalls else 0.0


## Baseline


In [ ]:
baseline_artifacts = fit_als(rank_history, n_users)
baseline_candidates = build_candidates(
    rank_target["user_idx"].unique(),
    rank_history,
    baseline_artifacts,
    k=500
)
print("Baseline Recall@500:", recall(baseline_candidates, rank_target))


## Candidate K


In [ ]:
candidates_2000 = build_candidates(
    rank_target["user_idx"].unique(),
    rank_history,
    baseline_artifacts,
    k=2000
).sort_values(["user_id", "score"], ascending=[True, False])

ks = np.arange(100, 2001, 100)
recall_df = pd.DataFrame({
    "k": ks,
    "recall": [
        recall(candidates_2000.groupby("user_id", sort=False).head(k), rank_target)
        for k in ks
    ]
})
recall_df["recall_pct_step"] = recall_df["recall"].pct_change() * 100
recall_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(recall_df["k"], recall_df["recall"])
ax.set(title="ALS candidate recall", xlabel="K", ylabel="Recall@K")
ax.grid(True)
plt.show()


## Parameter search


In [ ]:
factors_grid = [8, 16, 24, 25, 32, 40]
regularization_grid = [0.1, 0.2, 0.3, 0.5, 1.0]
iterations_grid = [10, 20, 30]

results = []

for factors, reg, iterations in product(
    factors_grid, regularization_grid, iterations_grid
):
    start = time.time()
    artifacts = fit_als(
        rank_history,
        n_users,
        factors=factors,
        regularization=reg,
        iterations=iterations
    )
    candidates = build_candidates(
        rank_target["user_idx"].unique(),
        rank_history,
        artifacts,
        k=500
    )
    results.append({
        "factors": factors,
        "regularization": reg,
        "iterations": iterations,
        "recall": recall(candidates, rank_target),
        "run_time": time.time() - start
    })

results_df = pd.DataFrame(results).sort_values("recall", ascending=False).reset_index(drop=True)
results_df.head(15)


## Selected model


In [ ]:
ALS_PARAMS = {
    "factors": 16,
    "regularization": 0.5,
    "iterations": 10,
}

selected_artifacts = fit_als(rank_history, n_users, **ALS_PARAMS)
rank_als_candidates = build_candidates(
    rank_target["user_idx"].unique(),
    rank_history,
    selected_artifacts,
    k=500
)

PARAM_DIR = ROOT_DIR / "notebook" / "data" / "parameters"
PARAM_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(ALS_PARAMS, PARAM_DIR / "00_best_als_parameters.pkl")
joblib.dump(rank_als_candidates, DATA_DIR / "als_candidates.pkl")

print("Selected Recall@500:", recall(rank_als_candidates, rank_target))


## Test baseline


In [ ]:
test_artifacts = fit_als(test_history, n_users, **ALS_PARAMS)
als_top10 = build_candidates(
    test_target["user_idx"].unique(),
    test_history,
    test_artifacts,
    k=10
)

y_true = (
    test_target.loc[test_target["rating"] >= 4]
    .groupby("user_id")["movie_id"]
    .apply(list)
    .to_dict()
)
y_pred = als_top10.groupby("user_id")["movie_id"].apply(list).to_dict()

recalls, ndcgs = [], []
for user_id, true_movies in y_true.items():
    pred_movies = y_pred.get(user_id, [])
    if len(pred_movies) < 10:
        continue
    recalls.append(recall_at_k(np.array(true_movies), np.array(pred_movies), k=10))
    ndcgs.append(ndcg_at_k(np.array(true_movies), np.array(pred_movies), k=10))

print("ALS Recall@10:", np.mean(recalls))
print("ALS NDCG@10:", np.mean(ndcgs))
print("Evaluated users:", len(ndcgs))
